# Inventory Data Cleaning

**Purpose:** prepare stock records for inventory analysis. This notebook standardizes stock values and highlights records that need attention.

**Expected columns:** product/item, quantity on hand, and optionally reorder level, unit cost, or warehouse.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path('data/inventory_raw.csv')  # Change this if your file has another name
OUTPUT_FILE = Path('output/inventory_cleaned.csv')
OUTPUT_FILE.parent.mkdir(exist_ok=True)

df = pd.read_excel(DATA_FILE) if DATA_FILE.suffix.lower() in {'.xlsx', '.xls'} else pd.read_csv(DATA_FILE)
print(f'Loaded {len(df):,} inventory rows')
df.head()

In [ ]:
# Make column names and text fields consistent
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_'))
for column in df.select_dtypes(include='object').columns:
    df[column] = df[column].astype('string').str.strip()
print('Columns:', list(df.columns))

In [ ]:
# Convert key quantities to numeric values
rows_before = len(df)
df = df.drop_duplicates().copy()

# Update these names if the columns in your file are different
df['quantity_on_hand'] = pd.to_numeric(df['quantity_on_hand'], errors='coerce')
if 'reorder_level' in df.columns:
    df['reorder_level'] = pd.to_numeric(df['reorder_level'], errors='coerce')
if 'unit_cost' in df.columns:
    df['unit_cost'] = pd.to_numeric(df['unit_cost'], errors='coerce')
    df['inventory_value'] = df['quantity_on_hand'] * df['unit_cost']

In [ ]:
# Retain records with a product and valid stock quantity
df = df.dropna(subset=['product', 'quantity_on_hand'])
df['stock_status'] = 'Available'
df.loc[df['quantity_on_hand'] <= 0, 'stock_status'] = 'Out of stock'
if 'reorder_level' in df.columns:
    df.loc[(df['quantity_on_hand'] > 0) & (df['quantity_on_hand'] <= df['reorder_level']), 'stock_status'] = 'Reorder needed'

audit = pd.DataFrame({
    'measure': ['input rows', 'duplicates removed', 'missing values remaining', 'clean output rows'],
    'value': [rows_before, rows_before - len(df), int(df.isna().sum().sum()), len(df)]
})
display(audit)
df[['product', 'quantity_on_hand', 'stock_status']].head()

In [ ]:
df.to_csv(OUTPUT_FILE, index=False)
print(f'Saved cleaned inventory data to: {OUTPUT_FILE}')